# M1 · Laboratorio de comportamiento

Repetís con un **LLM real** lo del cuaderno de Python (`m1_python_de_cero.ipynb`): cambiás **una** cosa y mirás qué pasa.

**La tarea, en tres pasos:** 1. **Predecí** (en `PREDICCION`, antes de correr) · 2. **Corré** · 3. **Anotá** el resultado antes de cambiar otra perilla.

- Todo corre acá, en Colab: no hay archivos ni carpetas.
- Corré las celdas en orden con **Shift + Enter**. La clave: **Secretos** (la llave de la izquierda) → `GOOGLE_API_KEY`.

<details><summary><b>¿Sale un error?</b></summary>

- **«No pude leer el secreto GOOGLE_API_KEY»** o **`API key not valid`:** revisá el nombre del secreto (exacto, en mayúsculas),
  que «Acceso del notebook» esté activado y que la clave esté bien copiada.
- **`503 UNAVAILABLE`** («high demand»): el modelo está saturado, no es tu código. Esperá un minuto y volvé a correr.
- **`429`** (*quota*, *RESOURCE_EXHAUSTED*): se terminó la cuota gratis del día o del minuto. Esperá o seguí mañana.
- **`NameError`:** salteaste una celda o Colab se reinició. Menú *Entorno de ejecución* → *Run before*.
- **«Tope de 20 llamadas»:** ya usaste las llamadas del laboratorio. Anotá lo que tenés: alcanza para explicar.
</details>


## 1 · Preparar (una vez, menos de un minuto)

In [ ]:
%pip install -q langchain==1.4.6 langchain-google-genai==4.4.2
import os

try:
    from google.colab import userdata                      # en Colab: la clave sale de los Secretos
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass                                                   # fuera de Colab: GOOGLE_API_KEY ya tiene que estar
except Exception as error:                                 # no hay secreto con ese nombre o le falta el acceso
    print("No pude leer el secreto GOOGLE_API_KEY:", type(error).__name__)

import time

import pandas as pd
from langchain.chat_models import init_chat_model

MODELO = "google_genai:gemini-3.5-flash-lite"
gastadas = 0  # llamadas usadas: acá y no en las perillas, así no vuelve a 0 al cambiar la temperatura
print("Listo." if os.environ.get("GOOGLE_API_KEY") else "Falta la clave: Secretos → GOOGLE_API_KEY.")

## 2 · Perillas

- `MODELO_T`: la temperatura la usa Gemini 3.1 Flash-Lite; el 3.5 la ignora.
- Tope: 20 llamadas por persona (`TOPE`). Cada repetición es una llamada.

In [ ]:
MODELO_T = "google_genai:gemini-3.1-flash-lite"
PROMPT = "Inventá un nombre para una cafetería en Montevideo. Respondé solo el nombre."
TEMPERATURA = 0.0
REPETICIONES = 5
TOPE = 20

## 3 · Experimento 1: temperatura

**Predecí antes:** con 0, ¿cuántas distintas sobre 5? Corré, anotá, cambiá `TEMPERATURA` a 1 en las perillas y corré de nuevo esa celda y la de `correr(...)`.

In [ ]:
def correr(prompt, temperatura, repeticiones, modelo_nombre=MODELO_T):
    global gastadas
    if gastadas + repeticiones > TOPE:
        print(f"Tope de {TOPE} llamadas: ya usaste {gastadas}.")
        return None
    modelo = init_chat_model(modelo_nombre, temperature=temperatura)
    filas = []
    for _ in range(repeticiones):
        try:
            r = modelo.invoke(prompt)
        except Exception as error:  # 503: el modelo está saturado; esperá un minuto y volvé a correr
            print("No respondió:", type(error).__name__,
                  "· si el error dice 503 o «high demand», Gemini está saturado: esperá un minuto y corré de nuevo.")
            break
        gastadas += 1
        filas.append({"respuesta": r.text.strip(), "entrada": r.usage_metadata["input_tokens"], "salida": r.usage_metadata["output_tokens"]})
    tabla = pd.DataFrame(filas)
    if len(tabla):
        print(f"T={temperatura}: {tabla['respuesta'].nunique()} distintas de {len(tabla)} · llamadas usadas: {gastadas}")
    return tabla

In [ ]:
PREDICCION = ""  # antes de correr: con temperatura 0, ¿cuántas distintas de 5? Por ejemplo: "1, todas iguales"


In [ ]:
print("Tu predicción:", PREDICCION or "(todavía no la escribiste: hacelo antes de correr)")
correr(PROMPT, TEMPERATURA, REPETICIONES)

**Qué deberías ver:** la tabla de respuestas y `T=0.0: 1 distintas de 5` (con T = 1, más).

**Primer control · 10–15 min:** la tabla con respuestas y tokens.

## 4 · Experimento 2: idioma

La misma frase en español y en inglés: ¿cuántos tokens de entrada usa cada una? Una llamada cada una.

In [ ]:
ES = "Los vecinos desconfiaban del nuevo plan de transporte que presentó la intendencia la semana pasada."
EN = "The neighbors distrusted the new transportation plan that the city presented last week."
if gastadas + 2 > TOPE:  # el mismo tope que el experimento 1: correr esta celda de nuevo también gasta
    print(f"Tope de {TOPE} llamadas: ya usaste {gastadas}.")
else:
    modelo = init_chat_model(MODELO)
    for frase in (ES, EN):
        r = modelo.invoke(frase)
        gastadas += 1
        print(f"{r.usage_metadata['input_tokens']} tokens de entrada · {len(frase.split())} palabras · {frase[:40]}…")

**Qué deberías ver:** el español usa más tokens que el inglés por la misma idea.

## 5 · Resultado y para casa

- Una fila: **predicción · resultado · explicación** (con el cuaderno de Python: softmax, temperatura, tokens).
- Para casa: un dato al principio, medio o final de un texto largo pegado en `PROMPT` (¿lo encuentra?), o algo que no existe, con y sin «si no sabés, decilo».

## Entregar

- **Guardá este notebook en tu repo:** *Archivo → Guardar una copia en GitHub* → tu repositorio → `m1.ipynb`.
- **Evidencia** (una celda de texto al final, acá mismo): la fila del experimento, una frase «Esto **no** demuestra que…» y lo que te imprimió el diagnóstico del cuaderno de Python.

**Microdefensa:** ¿Qué contradijo tu predicción? ¿Qué lo explicaría? Confianza del 1 al 4.